<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/main/src/notebooks/08_Pseudo-Labeling_CV_Domain_Seniority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score


In [ ]:
ANNOT = Path("linkedin-cvs-annotated.json")
UNLAB = Path("linkedin-cvs-not-annotated.json")

assert ANNOT.exists(), f"Missing file: {ANNOT}"
assert UNLAB.exists(), f"Missing file: {UNLAB}"

with open(ANNOT, "r", encoding="utf-8") as f:
    profiles_annot = json.load(f)

with open(UNLAB, "r", encoding="utf-8") as f:
    profiles_unlab = json.load(f)

# flatten list-of-lists if needed
if isinstance(profiles_annot, list) and len(profiles_annot) > 0 and isinstance(profiles_annot[0], list):
    profiles_annot = [x for sub in profiles_annot for x in sub]

if isinstance(profiles_unlab, list) and len(profiles_unlab) > 0 and isinstance(profiles_unlab[0], list):
    profiles_unlab = [x for sub in profiles_unlab for x in sub]

print("Annotated profiles:", len(profiles_annot))
print("Unlabeled profiles:", len(profiles_unlab))
print("Example keys (annotated):", list(profiles_annot[0].keys())[:25])


Annotated profiles: 2638
Unlabeled profiles: 1886
Example keys (annotated): ['organization', 'linkedin', 'position', 'startDate', 'endDate', 'status', 'department', 'seniority']


In [ ]:
def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, dict):
        x = " ".join([str(v) for v in x.values() if v is not None])
    elif isinstance(x, (list, tuple)):
        x = " ".join([str(v) for v in x if v is not None])
    else:
        x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def safe_get(d, keys, default=None):
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

def build_text(profile):
    # core current-job fields
    pos   = clean_text(safe_get(profile, ["position", "title", "jobTitle"], ""))
    org   = clean_text(safe_get(profile, ["organization", "company", "employer"], ""))
    stat  = clean_text(safe_get(profile, ["status"], ""))
    sd    = clean_text(safe_get(profile, ["startDate", "start_date"], ""))
    ed    = clean_text(safe_get(profile, ["endDate", "end_date"], ""))

    # extra context
    headline = clean_text(safe_get(profile, ["headline"], ""))
    summary  = clean_text(safe_get(profile, ["summary", "about"], ""))

    skills   = clean_text(safe_get(profile, ["skills", "skill"], ""))
    desc     = clean_text(safe_get(profile, ["description", "job_description", "jobDescription"], ""))

    industry = clean_text(safe_get(profile, ["industry"], ""))
    location = clean_text(safe_get(profile, ["location"], ""))

    linkedin = safe_get(profile, ["linkedin", "url", "profileUrl"], "")
    linkedin_txt = clean_text(linkedin)

    parts = [
        f"position: {pos}" if pos else "",
        f"org: {org}" if org else "",
        f"headline: {headline}" if headline else "",
        f"summary: {summary}" if summary else "",
        f"skills: {skills}" if skills else "",
        f"desc: {desc}" if desc else "",
        f"industry: {industry}" if industry else "",
        f"location: {location}" if location else "",
        f"start: {sd}" if sd else "",
        f"end: {ed}" if ed else "",
        f"status: {stat}" if stat else "",
        f"linkedin: {linkedin_txt}" if linkedin_txt else "",
    ]
    parts = [p for p in parts if p]
    return " | ".join(parts)


In [ ]:
def get_labels(profile):
    dom = safe_get(profile, ["department", "domain"], None)
    sen = safe_get(profile, ["seniority", "seniority_level"], None)

    labels = safe_get(profile, ["labels", "annotation", "annotations"], None)
    if isinstance(labels, dict):
        dom = dom or safe_get(labels, ["department", "domain"], None)
        sen = sen or safe_get(labels, ["seniority", "seniority_level"], None)

    return dom, sen

rows = []
for i, p in enumerate(profiles_annot):
    dom, sen = get_labels(p)
    rows.append({
        "idx": i,
        "text": build_text(p),
        "domain": clean_text(dom) if dom is not None else None,
        "seniority": clean_text(sen) if sen is not None else None,
        "status": clean_text(safe_get(p, ["status"], "")),
    })

df_gold = pd.DataFrame(rows)

df_gold = df_gold[
    (df_gold["text"].str.len() > 0) &
    (df_gold["domain"].notna()) &
    (df_gold["seniority"].notna())
].copy()

print("Gold usable rows:", len(df_gold))
print("\nGold domain distribution (top 10):")
print(df_gold["domain"].value_counts().head(10))
print("\nGold seniority distribution:")
print(df_gold["seniority"].value_counts())
df_gold.head()


Gold usable rows: 2638

Gold domain distribution (top 10):
domain
Other                     1252
Information Technology     312
Sales                      219
Consulting                 195
Project Management         175
Marketing                  133
Administrative              84
Business Development        78
Purchasing                  72
Human Resources             70
Name: count, dtype: int64

Gold seniority distribution:
seniority
Professional    1219
Lead             460
Management       418
Junior           230
Senior           170
Director         141
Name: count, dtype: int64


,idx,text,domain,seniority,status
0,0,position: Prokurist | org: Depot4Design GmbH |...,Other,Management,ACTIVE
1,1,position: CFO | org: Depot4Design GmbH | start...,Other,Management,ACTIVE
2,2,position: Betriebswirtin | org: Depot4Design G...,Other,Professional,ACTIVE
3,3,position: Prokuristin | org: Depot4Design GmbH...,Other,Management,ACTIVE
4,4,position: CFO | org: Depot4Design GmbH | start...,Other,Management,ACTIVE


In [ ]:
rows_un = []
for i, p in enumerate(profiles_unlab):
    rows_un.append({
        "idx": i,
        "text": build_text(p),
        "status": clean_text(safe_get(p, ["status"], "")),
    })

df_un = pd.DataFrame(rows_un)
df_un = df_un[df_un["text"].str.len() > 0].copy()

print("Unlabeled usable rows:", len(df_un))
df_un.head()


Unlabeled usable rows: 1886


,idx,text,status
0,0,"position: Bookkeeper | org: Keeping The Books,...",ACTIVE
1,1,position: Co-Owner | org: Playful Paws | start...,ACTIVE
2,2,position: Logistics Officer | org: S&R service...,INACTIVE
3,3,position: Truck driver/ laborer | org: ABC Sup...,INACTIVE
4,4,position: Fuel Driver | org: MB Railways | sta...,INACTIVE


In [ ]:
train_gold, val_gold = train_test_split(
    df_gold,
    test_size=0.2,
    random_state=42,
    stratify=df_gold["domain"]
)

print("train_gold:", len(train_gold), "val_gold:", len(val_gold))


train_gold: 2110 val_gold: 528


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9)

X_tr = tfidf.fit_transform(train_gold["text"])
X_va = tfidf.transform(val_gold["text"])

teacher_dom = LogisticRegression(max_iter=3000, class_weight="balanced")
teacher_dom.fit(X_tr, train_gold["domain"])

teacher_sen = LogisticRegression(max_iter=3000, class_weight="balanced")
teacher_sen.fit(X_tr, train_gold["seniority"])

pred_dom_va = teacher_dom.predict(X_va)
pred_sen_va = teacher_sen.predict(X_va)

print("TEACHER DOMAIN accuracy:", accuracy_score(val_gold["domain"], pred_dom_va))
print("TEACHER DOMAIN macro-F1:", f1_score(val_gold["domain"], pred_dom_va, average="macro"))
print("\nTEACHER DOMAIN report:\n", classification_report(val_gold["domain"], pred_dom_va))

print("TEACHER SENIORITY accuracy:", accuracy_score(val_gold["seniority"], pred_sen_va))
print("TEACHER SENIORITY macro-F1:", f1_score(val_gold["seniority"], pred_sen_va, average="macro"))
print("\nTEACHER SENIORITY report:\n", classification_report(val_gold["seniority"], pred_sen_va))


TEACHER DOMAIN accuracy: 0.5738636363636364
TEACHER DOMAIN macro-F1: 0.516029049517747

TEACHER DOMAIN report:
                         precision    recall  f1-score   support

        Administrative       0.10      0.24      0.14        17
  Business Development       0.53      0.62      0.57        16
            Consulting       0.68      0.59      0.63        39
      Customer Support       0.38      0.60      0.46        10
       Human Resources       0.39      0.64      0.49        14
Information Technology       0.67      0.71      0.69        62
             Marketing       0.33      0.63      0.43        27
                 Other       0.80      0.53      0.63       250
    Project Management       0.54      0.63      0.58        35
            Purchasing       0.30      0.64      0.41        14
                 Sales       0.69      0.61      0.65        44

              accuracy                           0.57       528
             macro avg       0.49      0.59      0.52 

In [ ]:
def pseudo_label(df_unlabeled, tfidf_vec, model_dom, model_sen,
                 conf_dom=0.80, conf_sen=0.80,
                 exclude_other=True, other_label="Other",
                 topk_per_domain=50):
    """
    Returns a pseudo-labeled dataframe with high confidence + balancing.
    """
    X_un = tfidf_vec.transform(df_unlabeled["text"])

    dom_proba = model_dom.predict_proba(X_un)
    dom_ids = dom_proba.argmax(axis=1)
    dom_conf = dom_proba.max(axis=1)
    dom_labels = model_dom.classes_

    sen_proba = model_sen.predict_proba(X_un)
    sen_ids = sen_proba.argmax(axis=1)
    sen_conf = sen_proba.max(axis=1)
    sen_labels = model_sen.classes_

    pseudo = df_unlabeled.copy()
    pseudo["pseudo_domain"] = dom_labels[dom_ids]
    pseudo["dom_conf"] = dom_conf
    pseudo["pseudo_seniority"] = sen_labels[sen_ids]
    pseudo["sen_conf"] = sen_conf

    # threshold filter
    keep = pseudo[(pseudo["dom_conf"] >= conf_dom) & (pseudo["sen_conf"] >= conf_sen)].copy()

    # exclude "Other" from pseudo-training if desired
    if exclude_other:
        keep = keep[keep["pseudo_domain"] != other_label].copy()

    # top-k balancing per domain class
    balanced = []
    for c in keep["pseudo_domain"].unique():
        sub = keep[keep["pseudo_domain"] == c].sort_values("dom_conf", ascending=False).head(topk_per_domain)
        balanced.append(sub)

    if len(balanced) > 0:
        keep = pd.concat(balanced, ignore_index=True)

    return keep


In [ ]:
N_ITERS = 5
CONF_DOM = 0.40
CONF_SEN = 0.40
TOPK_PER_DOMAIN = 100

EXCLUDE_OTHER = True
OTHER_LABEL = "Other"

work_train = train_gold.copy()

history = []

for it in range(1, N_ITERS + 1):
    print(f"\n================ ITERATION {it}/{N_ITERS} ================")

    # 1) Train teacher on current training mix
    tfidf_it = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9)
    X_mix = tfidf_it.fit_transform(work_train["text"])
    X_val = tfidf_it.transform(val_gold["text"])

    model_dom = LogisticRegression(max_iter=3000, class_weight="balanced")
    model_dom.fit(X_mix, work_train["domain"])

    model_sen = LogisticRegression(max_iter=3000, class_weight="balanced")
    model_sen.fit(X_mix, work_train["seniority"])

    # 2) Evaluate on gold validation
    pred_dom = model_dom.predict(X_val)
    pred_sen = model_sen.predict(X_val)

    dom_acc = accuracy_score(val_gold["domain"], pred_dom)
    dom_f1  = f1_score(val_gold["domain"], pred_dom, average="macro")

    sen_acc = accuracy_score(val_gold["seniority"], pred_sen)
    sen_f1  = f1_score(val_gold["seniority"], pred_sen, average="macro")

    print(f"VAL DOMAIN  acc={dom_acc:.3f} macroF1={dom_f1:.3f}")
    print(f"VAL SENIOR  acc={sen_acc:.3f} macroF1={sen_f1:.3f}")

    history.append((it, len(work_train), dom_acc, dom_f1, sen_acc, sen_f1))

    # 3) Pseudo-label unlabeled with high confidence
    pseudo = pseudo_label(
        df_un,
        tfidf_it,
        model_dom,
        model_sen,
        conf_dom=CONF_DOM,
        conf_sen=CONF_SEN,
        exclude_other=EXCLUDE_OTHER,
        other_label=OTHER_LABEL,
        topk_per_domain=TOPK_PER_DOMAIN
    )

    print("Pseudo-labeled kept:", len(pseudo))
    if len(pseudo) == 0:
        print("No pseudo labels passed thresholds. Stopping early.")
        break

    # 4) Add pseudo labels to training mix
    pseudo_for_train = pseudo.rename(columns={
        "pseudo_domain": "domain",
        "pseudo_seniority": "seniority"
    })[["text","domain","seniority"]].copy()

    work_train = pd.concat([work_train, pseudo_for_train], ignore_index=True)

print("\nTraining history:")
hist_df = pd.DataFrame(history, columns=["iter","train_size","dom_acc","dom_macroF1","sen_acc","sen_macroF1"])
display(hist_df)


================ ITERATION 1/5 ================
VAL DOMAIN  acc=0.574 macroF1=0.516
VAL SENIOR  acc=0.775 macroF1=0.775
Pseudo-labeled kept: 193

================ ITERATION 2/5 ================
VAL DOMAIN  acc=0.597 macroF1=0.528
VAL SENIOR  acc=0.775 macroF1=0.777
Pseudo-labeled kept: 245

================ ITERATION 3/5 ================
VAL DOMAIN  acc=0.612 macroF1=0.539
VAL SENIOR  acc=0.780 macroF1=0.782
Pseudo-labeled kept: 272

================ ITERATION 4/5 ================
VAL DOMAIN  acc=0.625 macroF1=0.544
VAL SENIOR  acc=0.782 macroF1=0.787
Pseudo-labeled kept: 290

================ ITERATION 5/5 ================
VAL DOMAIN  acc=0.640 macroF1=0.554
VAL SENIOR  acc=0.784 macroF1=0.788
Pseudo-labeled kept: 312

Training history:


,iter,train_size,dom_acc,dom_macroF1,sen_acc,sen_macroF1
0,1,2110,0.573864,0.516029,0.774621,0.774736
1,2,2303,0.596591,0.528000,0.774621,0.776984
2,3,2548,0.611742,0.538896,0.780303,0.781592
3,4,2820,0.625000,0.543623,0.782197,0.787208
4,5,3110,0.640152,0.554149,0.784091,0.788312


In [ ]:
tfidf_final = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9)
X_final = tfidf_final.fit_transform(work_train["text"])
X_un = tfidf_final.transform(df_un["text"])

final_dom = LogisticRegression(max_iter=3000, class_weight="balanced")
final_dom.fit(X_final, work_train["domain"])

final_sen = LogisticRegression(max_iter=3000, class_weight="balanced")
final_sen.fit(X_final, work_train["seniority"])

df_pred = df_un.copy()
df_pred["pred_domain_approach4"] = final_dom.predict(X_un)
df_pred["pred_seniority_approach4"] = final_sen.predict(X_un)

print("Pred domain distribution (top 15):")
display(df_pred["pred_domain_approach4"].value_counts().head(15))

print("Pred seniority distribution:")
display(df_pred["pred_seniority_approach4"].value_counts())

Pred domain distribution (top 15):


,count
pred_domain_approach4,
Other,912
Information Technology,177
Project Management,118
Consulting,117
Sales,117
Purchasing,103
Marketing,99
Administrative,73
Business Development,70


Pred seniority distribution:


,count
pred_seniority_approach4,
Professional,768
Management,341
Lead,294
Junior,265
Director,111
Senior,107


In [ ]:
OUTFILE = "predictions_approach4_selftraining.csv"
df_pred[["idx","pred_domain_approach4","pred_seniority_approach4"]].to_csv(OUTFILE, index=False)
print("Saved:", OUTFILE)

Saved: predictions_approach4_selftraining.csv
